<a href="https://colab.research.google.com/github/NETHRA-S-CSE/Agentic_AI_Internship/blob/main/AgenticAI_Task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kagglehub

In [ ]:
import kagglehub
ai_human_text_dataset_path=kagglehub.dataset_download("shanegerami/ai-vs-human-text")

Using Colab cache for faster access to the 'ai-vs-human-text' dataset.


In [ ]:
import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
  for filename in filenames:
    print(os.path.join(dirname, filename))


/kaggle/input/ai-vs-human-text/AI_Human.csv


In [ ]:
import re ## for regular experssions
import string ## for string operations
import numpy as np ##
import random
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
from plotly import graph_objs as go
import plotly.express as px
import plotly.figure_factory as ff
from collections import Counter

from PIL import Image
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator


import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from tqdm import tqdm
import os
import nltk
import spacy
import random
from spacy.util import compounding
from spacy.util import minibatch

from collections import defaultdict
from collections import Counter

import keras
from keras.models import Sequential
from keras.initializers import Constant
from keras.layers import (LSTM,
                          Embedding,
                          BatchNormalization,
                          Dense,
                          TimeDistributed,
                          Dropout,
                          Bidirectional,
                          Flatten,
                          GlobalMaxPool1D)
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.layers import Embedding
from keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from keras.optimizers import Adam

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    accuracy_score
)

In [ ]:
df=pd.read_csv("/kaggle/input/ai-vs-human-text/AI_Human.csv",encoding="latin=1")

df=df.dropna(how="any",axis=1)
df.columns=['text','target']


In [ ]:
balance_counts= df['target'].value_counts()
balance_counts

,count
target,
0.0,305797
1.0,181438


In [ ]:
!pip install -U plotly

In [ ]:
primary_blue = "#496595"
primary_blue2 = "#85a1s1"
primary_blue3 = "#3f4d63"
primary_grey = "#c6ccd8"
primary_black = "#202022"
primary_bgcolor = "#f4f0ea"
primary_green = px.colors.qualitative.Plotly[2]

In [ ]:
import plotly.io as pio
pio.renderers.default = "colab"

fig = go.Figure()
fig.add_trace(go.Bar(
    x=['0.0'],
    y=[balance_counts.iloc[0]],
    name='AI',
    text=[balance_counts.iloc[0]],
    textposition='auto',
    marker_color=primary_blue
))
fig.add_trace(go.Bar(
    x=['1.0'],
    y=[balance_counts.iloc[1]],
    name='human',
    text=[balance_counts.iloc[1]],
    textposition='auto',
    marker_color=primary_grey
))
fig.update_layout(
    title='<span style="font-size:32px; font-family:Times New Roman">Dataset distribution by target</span>'
)
fig.show(renderer="colab")

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

<>:3: SyntaxWarning: invalid escape sequence '\['
<>:4: SyntaxWarning: invalid escape sequence '\S'
<>:8: SyntaxWarning: invalid escape sequence '\w'
<>:3: SyntaxWarning: invalid escape sequence '\['
<>:4: SyntaxWarning: invalid escape sequence '\S'
<>:8: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_7557/2282808393.py:3: SyntaxWarning: invalid escape sequence '\['
  text = re.sub('\[.*?\]', '', text)
/tmp/ipykernel_7557/2282808393.py:4: SyntaxWarning: invalid escape sequence '\S'
  text = re.sub('https?://\S+|www\.\S+', '', text)
/tmp/ipykernel_7557/2282808393.py:8: SyntaxWarning: invalid escape sequence '\w'
  text = re.sub('\w*\d\w*', '', text)


In [ ]:
df['text_clean'] = df['text'].apply(clean_text)
df.head(15)

,text,target,text_clean
0,Cars. Cars have been around since they became ...,0.0,cars cars have been around since they became f...
1,Transportation is a large necessity in most co...,0.0,transportation is a large necessity in most co...
2,"""America's love affair with it's vehicles seem...",0.0,americas love affair with its vehicles seems t...
3,How often do you ride in a car? Do you drive a...,0.0,how often do you ride in a car do you drive a ...
4,Cars are a wonderful thing. They are perhaps o...,0.0,cars are a wonderful thing they are perhaps on...
5,The electrol college system is an unfair syste...,0.0,the electrol college system is an unfair syste...
6,"Dear state senator, It is the utmost respect t...",0.0,dear state senator it is the utmost respect th...
7,"Fellow citizens, cars have become a major role...",0.0,fellow citizens cars have become a major role ...
8,"""It's official: The electoral college is unfai...",0.0,its official the electoral college is unfair o...
9,The Electoral College has been kept for centur...,0.0,the electoral college has been kept for centur...


In [ ]:
import nltk
nltk.download('stopwords')
stop_words = stopwords.words('english')

def remove_stopwords(text):
    text = ' '.join(word for word in text.split(' ') if word not in stop_words)
    return text

df['text_clean'] = df['text_clean'].apply(remove_stopwords)
df.head(15)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,text,target,text_clean
0,Cars. Cars have been around since they became ...,0.0,cars cars around since became famous henry fo...
1,Transportation is a large necessity in most co...,0.0,transportation large necessity countries world...
2,"""America's love affair with it's vehicles seem...",0.0,americas love affair vehicles seems cooling sa...
3,How often do you ride in a car? Do you drive a...,0.0,often ride car drive one motor vehicle work st...
4,Cars are a wonderful thing. They are perhaps o...,0.0,cars wonderful thing perhaps one worlds greate...
5,The electrol college system is an unfair syste...,0.0,electrol college system unfair system people d...
6,"Dear state senator, It is the utmost respect t...",0.0,dear state senator utmost respect ask method p...
7,"Fellow citizens, cars have become a major role...",0.0,fellow citizens cars become major role daily l...
8,"""It's official: The electoral college is unfai...",0.0,official electoral college unfair outdated irr...
9,The Electoral College has been kept for centur...,0.0,electoral college kept centuries established f...


In [ ]:
stemmer=nltk.SnowballStemmer("english")

def stemm_text(text):
  text=' '.join(stemmer.stem(word) for word in text.split(' '))
  return text


In [ ]:
df['text_clean']=df['text_clean'].apply(stemm_text)
df.head()

,text,target,text_clean
0,Cars. Cars have been around since they became ...,0.0,car car around sinc becam famous henri ford c...
1,Transportation is a large necessity in most co...,0.0,transport larg necess countri worldwid doubt c...
2,"""America's love affair with it's vehicles seem...",0.0,america love affair vehicl seem cool say elisa...
3,How often do you ride in a car? Do you drive a...,0.0,often ride car drive one motor vehicl work sto...
4,Cars are a wonderful thing. They are perhaps o...,0.0,car wonder thing perhap one world greatest adv...


In [ ]:
def preprocess_data(text):
  text=clean_text(text)
  text=' '.join(word for word in text.split(' ') if word not in stop_words)
  text=' '.join(stemmer.stem(word) for word in text.split(' '))
  return text

In [ ]:
df['text_clean']=df['text_clean'].apply(preprocess_data)
df.head(15)

,text,target,text_clean
0,Cars. Cars have been around since they became ...,0.0,car car around sinc becam famous henri ford c...
1,Transportation is a large necessity in most co...,0.0,transport larg necess countri worldwid doubt c...
2,"""America's love affair with it's vehicles seem...",0.0,america love affair vehicl seem cool say elisa...
3,How often do you ride in a car? Do you drive a...,0.0,often ride car drive one motor vehicl work sto...
4,Cars are a wonderful thing. They are perhaps o...,0.0,car wonder thing perhap one world greatest adv...
5,The electrol college system is an unfair syste...,0.0,electrol colleg system unfair system peopl don...
6,"Dear state senator, It is the utmost respect t...",0.0,dear state senat utmost respect ask method pre...
7,"Fellow citizens, cars have become a major role...",0.0,fellow citizen car becom major role daili live...
8,"""It's official: The electoral college is unfai...",0.0,offici elector colleg unfair outdat irrat plum...
9,The Electoral College has been kept for centur...,0.0,elector colleg kept centuri establish found fa...


In [ ]:
df['target'] = df['target'].astype(int)

In [ ]:
x=df['text_clean']
y=df['target']
print(len(x),len(y))

487235 487235


In [ ]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=42,test_size=0.2)
print(len(x_train),len(y_train))
print(len(x_test),len(y_test))

389788 389788
97447 97447


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vect=CountVectorizer() ## model selection
vect.fit(x_train) ## to train

CountVectorizer()

In [ ]:
x_train_dtm=vect.transform(x_train)
x_test_dtm=vect.transform(x_test)
print(x_train_dtm.shape)
print(x_test_dtm.shape)

(389788, 387997)
(97447, 387997)


In [ ]:
vect_tunned=CountVectorizer(stop_words='english',ngram_range=(1,2),min_df=0.1,max_df=0.7,max_features=100)

In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer
tfidf_transformer=TfidfTransformer()
tfidf_transformer.fit(x_train_dtm)
x_train_tfidf=tfidf_transformer.transform(x_train_dtm)

x_train_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 44098146 stored elements and shape (389788, 387997)>

In [ ]:
texts=df['text_clean']
target=df['target']

In [ ]:
word_tokenizer=Tokenizer()
word_tokenizer.fit_on_texts(texts)
vocab_length=len(word_tokenizer.word_index)+1
vocab_length

461914